# Tokenization, Chat Templates, and Loss Masks

Fine-tuning quality depends heavily on formatting. A chat model is not trained on raw `instruction` and `response` fields directly. Those fields are converted into a token sequence using a model-specific chat template, then labels decide which tokens contribute to the loss.

This notebook shows the mechanics clearly before you use higher-level training libraries.

## What You Will Learn

By the end, you should understand:

- what tokenization does,
- why chat templates are model-specific,
- why assistant-only loss matters,
- what `input_ids`, `attention_mask`, and `labels` mean,
- how padding and truncation affect training,
- and how to debug a formatted fine-tuning example.

## 1. The Training Example Pipeline

A single fine-tuning example usually moves through this pipeline:

```text
raw record -> chat messages -> chat template string -> tokens -> labels -> batch
```

Every step can create bugs. The most common ones are training on user tokens, forgetting end-of-turn tokens, truncating the answer, or using a template that does not match the base model.

In [ ]:
import importlib.util
from dataclasses import dataclass

import pandas as pd

def has_package(name):
    return importlib.util.find_spec(name) is not None

print(f"transformers installed: {has_package('transformers')}")
print(f"torch installed: {has_package('torch')}")

## 2. Example Chat Record

We will use OpenAI-style messages because the structure is easy to read: each message has a role and content. Hugging Face tokenizers can often convert this structure with `apply_chat_template`, but the exact text format depends on the model.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise ML engineering tutor."},
    {"role": "user", "content": "Explain why assistant-only loss is used in SFT."},
    {"role": "assistant", "content": "Assistant-only loss trains the model on the desired answer while ignoring prompt tokens that were provided as conditioning context."},
]
messages

## 3. A Simple Chat Template

Real models use special tokens such as beginning-of-sequence, end-of-turn, or role tokens. For concept learning, this simple template is enough to show where roles appear and where the assistant answer begins.

In [ ]:
def simple_chat_template(messages):
    rendered = []
    for message in messages:
        role = message["role"].upper()
        rendered.append(f"<|{role}|>\n{message['content']}")
    rendered.append("<|END|>")
    return "\n".join(rendered)

formatted_text = simple_chat_template(messages)
print(formatted_text)

## 4. Use a Real Tokenizer When Available

If `transformers` is installed, this cell loads a tiny GPT-2 tokenizer for demonstration. GPT-2 is not a chat model, so we still use our simple text template. For real chat models, prefer the tokenizer's native chat template when it exists.

In [ ]:
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    encoded = tokenizer(formatted_text, return_tensors=None)
    print(f"vocab_size: {tokenizer.vocab_size}")
    print(f"num_tokens: {len(encoded['input_ids'])}")
    print(encoded['input_ids'][:20])
except Exception as error:
    tokenizer = None
    print("Tokenizer demo skipped. Install transformers to run it.")
    print(f"Reason: {error}")

## 5. A Tiny Concept Tokenizer

To make loss masking visible without depending on external packages, this tiny tokenizer splits on spaces and assigns integer IDs. It is not a real LLM tokenizer, but it is perfect for seeing how `input_ids` and `labels` align.

In [ ]:
class TinyWhitespaceTokenizer:
    def __init__(self):
        self.token_to_id = {"<pad>": 0, "<unk>": 1}
        self.id_to_token = {0: "<pad>", 1: "<unk>"}

    def fit(self, texts):
        for text in texts:
            for token in text.split():
                if token not in self.token_to_id:
                    index = len(self.token_to_id)
                    self.token_to_id[token] = index
                    self.id_to_token[index] = token

    def encode(self, text):
        return [self.token_to_id.get(token, 1) for token in text.split()]

    def decode(self, ids):
        return " ".join(self.id_to_token.get(index, "<unk>") for index in ids)

tiny_tokenizer = TinyWhitespaceTokenizer()
tiny_tokenizer.fit([formatted_text])
input_ids = tiny_tokenizer.encode(formatted_text)
print(input_ids)
print(tiny_tokenizer.decode(input_ids))

## 6. Assistant-Only Loss Masking

For supervised fine-tuning, the model should learn to produce the assistant answer, not memorize the system and user text. We create labels where non-assistant tokens are `-100`, which PyTorch ignores in cross entropy.

In [ ]:
def render_segments(messages):
    segments = []
    for message in messages:
        role = message["role"]
        text = f"<|{role.upper()}|>\n{message['content']}"
        segments.append((role, text))
    segments.append(("special", "<|END|>"))
    return segments

def tokenize_with_assistant_labels(messages, tokenizer):
    all_input_ids = []
    all_labels = []
    for role, text in render_segments(messages):
        ids = tokenizer.encode(text)
        all_input_ids.extend(ids)
        if role == "assistant":
            all_labels.extend(ids)
        else:
            all_labels.extend([-100] * len(ids))
    return all_input_ids, all_labels

input_ids, labels = tokenize_with_assistant_labels(messages, tiny_tokenizer)
rows = []
for token_id, label in zip(input_ids, labels):
    rows.append({
        "token": tiny_tokenizer.decode([token_id]),
        "input_id": token_id,
        "label": label,
        "trained": label != -100,
    })
pd.DataFrame(rows)

## 7. Padding and Attention Masks

Training happens in batches, but examples have different lengths. Padding makes them the same length. The `attention_mask` tells the model which positions are real tokens and which are padding. Labels at padding positions should also be ignored.

In [ ]:
def pad_to_length(input_ids, labels, max_length, pad_id=0):
    input_ids = input_ids[:max_length]
    labels = labels[:max_length]
    attention_mask = [1] * len(input_ids)
    pad_amount = max_length - len(input_ids)
    input_ids = input_ids + [pad_id] * pad_amount
    labels = labels + [-100] * pad_amount
    attention_mask = attention_mask + [0] * pad_amount
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }

batch_item = pad_to_length(input_ids, labels, max_length=36)
pd.DataFrame(batch_item).head(40)

## 8. Truncation Risks

If `max_length` is too short, you may truncate the answer. That means the model is trained on incomplete targets. Always inspect how much of each example is being truncated.

In [ ]:
def truncation_report(messages, tokenizer, max_lengths):
    input_ids, labels = tokenize_with_assistant_labels(messages, tokenizer)
    total_trained = sum(label != -100 for label in labels)
    rows = []
    for max_length in max_lengths:
        truncated_labels = labels[:max_length]
        trained_kept = sum(label != -100 for label in truncated_labels)
        rows.append({
            "max_length": max_length,
            "tokens_kept": min(max_length, len(input_ids)),
            "total_tokens": len(input_ids),
            "assistant_tokens_kept": trained_kept,
            "assistant_tokens_total": total_trained,
            "lost_answer_tokens": max(0, total_trained - trained_kept),
        })
    return pd.DataFrame(rows)

truncation_report(messages, tiny_tokenizer, [8, 16, 24, 36, 64])

## 9. Debugging Checklist

Before training, inspect a few formatted examples and answer these questions:

- Does the template match the model family?
- Is the assistant answer present?
- Are user and system tokens masked with `-100`?
- Are padding tokens masked?
- Is the answer truncated?
- Are special end-of-turn tokens present where the model expects them?
- Does the model have a pad token configured?
- Is right-padding or left-padding appropriate for the training code?

## Summary

Fine-tuning is not just model training. It is also formatting. Chat templates define what the model sees, tokenization turns text into IDs, labels define what loss is applied to, and masks prevent the model from training on prompts or padding.

Next, we will study full, partial, and parameter-efficient fine-tuning, then implement LoRA from scratch.